In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
from pathlib import Path
import random

In [ ]:
data_path = Path('../../data/interim/scenario_dataset_v1')

print(f"Checking path: {data_path}")
print(f"Exists: {data_path.exists()}")

if data_path.exists():
    # 1. Check the structural groupings
    partitions = sorted(list(data_path.glob('layout_id=*')))
    print(f"\nFound {len(partitions)} unique layout partitions")
    print(f"First 5 partitions: {[p.name for p in partitions[:5]]}")
    
    # 2. Check the raw file count
    total_files = len(list(data_path.rglob("*.parquet")))
    print(f"Total individual Parquet files across all partitions: {total_files}")
else:
    print("Path not found! Check your directory structure relative to this notebook.")

#### FINDINGS: 

* There are 10 layouts now
* each layout has 500 scenario
* so there is a total of 10 parquet files

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Grab the first actual parquet file from the first partition folder
first_partition_dir = partitions[0]
first_file = list(first_partition_dir.glob("*.parquet"))[0]

print(f"Inspecting file: {first_file.name}\n")

# --- THE DATA ENGINEER WAY (Instant Schema Check) ---
print("--- Parquet Schema (Metadata) ---")
schema = pq.read_schema(first_file)
print(schema)
print("-" * 40 + "\n")

# --- THE DATA SCIENTIST WAY (Visualizing the Data) ---
print("--- First 5 Rows ---")
df = pd.read_parquet(first_file)
display(df.head()) # 'display' looks cleaner than 'print' in Jupyter

#### FINDINGS

* Type: Comparative Scenario Simulation (Non-Temporal)
* Primary Key: Composite (layout_id, scenario_id)
* Observation: Initial data shows high variance in total_power relative to wind_speed.

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
print(f"Loaded DataFrame Shape: {df.shape}")
df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
import missingno as msno

msno.matrix(df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df["wind_speed"], bins=100)
plt.title("Wind Speed Distribution")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the features to plot
features = ["wind_speed", "wind_direction", "turbulence_intensity", "num_turbines"]

# Create a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Flatten axes for easy iteration
axes = axes.flatten()

for i, col in enumerate(features):
    sns.histplot(df[col], bins=100, ax=axes[i], kde=True)
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

# Adjust layout to prevent overlapping titles
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Features to check against the target 'total_power'
features = ["wind_speed", "wind_direction", "turbulence_intensity", "num_turbines"]

# Create a 2x2 grid for bivariate analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.scatterplot(x=col, y="total_power", data=df, ax=axes[i], alpha=0.5)
    axes[i].set_title(f"{col} vs. Total Power")
    axes[i].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Annotated Correlation Matrix for Power Output")
plt.show()

#### FINDINGS

* num_turbines is constant across all the scenarios so we can omit that during feature selection as it has minimal effect
* there is a strong correlation between wind_speed and total_power which is 0.84
* wind_direction and turbulence_intensity is uncorrelated with each other

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x="layout_id", y="total_power", data=df)
plt.xticks(rotation=90)
plt.title("Power Output Distribution per Layout")
plt.show()

In [ ]:
sns.scatterplot(x="scenario_id", y="total_power", data=df)
plt.title("Power Output by Scenario")
plt.show()

#### FINDINGS

Target Variable - The model will predict total_power.

* This represents the total wind plant power output for a given scenario.
It avoids turbine-level complexity and keeps the problem as tabular regression.

* Candidate Raw Features

    - layout_id
    - scenario_id
    - wind_speed
    - wind_direction
    - num_turbines

* Derived Feature Ideas
    - power_per_turbine = total_power / num_turbines
    - wind_direction_sin = sin(wind_direction)
    - wind_direction_cos = cos(wind_direction)
    - wind_speed_squared = wind_speed^2